# Benchmark breadth
Explicit one-dataset, one-stage development inference. Default plan only. See docs/benchmark_breadth_colab.md for source, math/scoring and privacy prerequisites. No locked cohort execution.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re
from google.colab import drive
drive.mount('/content/drive')
GIT_REF = "PASTE_REVIEWED_FULL_COMMIT_SHA"
REPO_URL = "https://github.com/Soqoro/Pact.git"
DATASET = "mmlu-pro"
STAGE = "plan"  # plan/preflight/smoke/private/score/revisions/readout/report/export
PHASE = "A"  # B must be separately selected for revisions/readout
MODEL = "Q"
DOWNLOAD = False  # explicit official source download
EXECUTE = False  # no model inference on default Run All
RESUME = False
ALLOW_PENDING_SCORING = False
SCORER_PYTHON = None
GPQA_PARENT = None  # verified parent data/partition JSON, outside Git
SCRATCH = Path('/content/pact-breadth')
CHECKOUT = SCRATCH/'checkout'
CACHE = SCRATCH/'cache'
SOURCE = SCRATCH/'sources'/DATASET
DATA = SCRATCH/'prepared'/(DATASET+'.json')
RUN = SCRATCH/'runs'/('pact-breadth-'+DATASET+'-001')
PERSISTENT = Path('/content/drive/MyDrive/PACT/breadth')/RUN.name
STOP_AFTER = None


In [ ]:
assert re.fullmatch(r'[0-9a-fA-F]{40}', GIT_REF)
SCRATCH.mkdir(parents=True,exist_ok=True)
def git(*args):
    r=subprocess.run(['git',*args],capture_output=True,text=True,env={**os.environ,'GIT_TERMINAL_PROMPT':'0'})
    if r.returncode:raise RuntimeError('Verify repository access and pushed revision.')
    return r.stdout.strip()
if not CHECKOUT.exists():git('clone','--filter=blob:none',REPO_URL,str(CHECKOUT))
assert not git('-C',str(CHECKOUT),'status','--porcelain'), 'Preserve local changes.'
assert git('-C',str(CHECKOUT),'remote','get-url','origin') == REPO_URL
git('-C',str(CHECKOUT),'fetch','--depth','1','origin',GIT_REF)
git('-C',str(CHECKOUT),'checkout','--detach',GIT_REF)
assert git('-C',str(CHECKOUT),'rev-parse','HEAD') == GIT_REF
os.chdir(CHECKOUT)
os.environ['PYTHONPATH']=str(CHECKOUT/'src')
os.environ['TOKENIZERS_PARALLELISM']='false'
subprocess.run([sys.executable,'-c','from pact.colab import install_dependencies; from pathlib import Path; install_dependencies(Path.cwd())'],check=True)
from google.colab import userdata
import getpass
try:token=userdata.get('HF_TOKEN')
except Exception:token=getpass.getpass('Authorized Hugging Face token (hidden): ')
if token:os.environ['HF_TOKEN']=token
token=None


In [ ]:
def command(*args):
    print('Starting:', *map(str,args),flush=True)
    with subprocess.Popen([sys.executable,'-u','-m','pact',*map(str,args)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as proc:
        for line in proc.stdout:print(line,end='',flush=True)
        code=proc.wait()
    if code:raise RuntimeError('Stage stopped; preserve files and return diagnostics. No automatic retry.')
def study(stage,*options):
    command('breadth',stage,'--run-dir',RUN,'--persistent',PERSISTENT,*options)
if RESUME and not RUN.exists():study('restore')
if not (RUN/'plan.json').exists():
    assert not list((PERSISTENT/'snapshots').glob('*')), 'Existing persistent study: set RESUME=True.'
    if not DATA.exists():
        args=['benchmarks','prepare','--dataset',DATASET,'--source-dir',SOURCE,'--output',DATA]
        prior=sorted(p for p in DATA.parent.glob('*.json') if p.name in {name+'.json' for name in ('mmlu-pro','musr','math-500','mbpp-plus','gpqa','livebench')} and p != DATA)
        if prior:
            exposure=SCRATCH/'exposure-index.json'
            command('benchmarks','exposure-index','--prepared',*prior,'--output',exposure)
            args+=['--exposure',exposure]
        if DOWNLOAD:args+=['--download']
        if GPQA_PARENT:args+=['--parent',GPQA_PARENT]
        command(*args)
    study('plan','--data',DATA)
r=subprocess.run([sys.executable,'-c','import sys; from pathlib import Path; from pact.studies.breadth import load; from pact.util import digest; print(digest(load(Path(sys.argv[1]))))',str(RUN)],check=True,capture_output=True,text=True)
PLAN_HASH=r.stdout.strip()
COMMON=['--plan-hash',PLAN_HASH,'--cache-dir',CACHE]
print('PLAN_HASH:',PLAN_HASH)
print('Budget:',json.loads((RUN/'budget_plan.json').read_text()))


In [ ]:
if STAGE == 'plan':
    print('Plan only. No model calls dispatched.')
elif STAGE in ('report','export'):
    study(STAGE)
elif STAGE == 'score':
    options=['--scorer-python',SCORER_PYTHON] if SCORER_PYTHON else []
    study('score',*COMMON,*options)
else:
    assert EXECUTE, 'Explicitly enable only your selected stage.'
    options=[]
    if STAGE in ('private','revisions'):options+=['--model',MODEL]
    if STOP_AFTER is not None:options+=['--stop-after',STOP_AFTER]
    if STAGE=='preflight':
        if SCORER_PYTHON:options+=['--scorer-python',SCORER_PYTHON]
        if ALLOW_PENDING_SCORING:options+=['--allow-pending-scoring']
    study(STAGE,*COMMON,'--execute','--phase',PHASE,*options)
